# FedAvg by error weight (frequency-weighted)

Aggregates client LoRA adapters using **frequency-weighted** Federated Averaging (formula from Progress_10thMarch.pdf):
- Error frequency: \(F_e = \sum_k \text{count}_{k,e}\)
- Error-type weight: \(\text{weight}_e = F_e / \sum_{e'} F_{e'}\)
- Client error score: \(E_k = \sum_e \text{count}_{k,e} \cdot \text{weight}_e\)
- Client weight: \(\alpha_k = E_k / \sum_j E_j\)
- Output: merged adapter at `aggregated_adapters/fedavg_error/`

## 1. Paths and client adapter discovery

In [1]:
# !pip install -q pandas safetensors torch

In [1]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path
from collections import defaultdict

try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

BASE_DIR = Path(NOTEBOOK_DIR)
CSV_PATH = BASE_DIR / "final_dataset_v2.csv"
OUTPUT_DIR = BASE_DIR / "aggregated_adapters" / "fedavg_error"

CLIENT_CONFIG = {
    "mbpp": BASE_DIR / "MBPP" / "lora_adapters_mbpp",
    "humaneval": BASE_DIR / "HUMANEVAL" / "lora_adapters_humaneval",
    "ds1000": BASE_DIR / "DS1000" / "lora_adapters_ds1000",
}

adapters_found = {k: v for k, v in CLIENT_CONFIG.items() if (v / "adapter_model.safetensors").exists()}
missing = [k for k in CLIENT_CONFIG if k not in adapters_found]
if missing:
    print("Skipping clients (no adapter_model.safetensors):", missing)
print("Clients to aggregate:", list(adapters_found.keys()))

Clients to aggregate: ['mbpp', 'humaneval', 'ds1000']


## 2. Compute error-type and client weights from final_dataset_v2.csv

In [2]:
import ast

df = pd.read_csv(CSV_PATH)
for col in ["dataset", "status", "ast_info", "dynamic_info", "lib_info"]:
    if col not in df.columns:
        raise ValueError(f"final_dataset_v2.csv must have '{col}' column")

df["_client"] = df["dataset"].astype(str).str.strip().str.lower()

def _safe_literal_eval(x):
    if x is None:
        return None
    # pandas NaN
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return None
    try:
        return ast.literal_eval(s)
    except Exception:
        return None


def _clean_label(x: str) -> str | None:
    s = str(x or "").strip()
    if not s:
        return None
    if s.lower() == "nan":
        return None
    return s


def parse_error_labels(row) -> set[str]:
    labels: set[str] = set()

    status = str(row.get("status", "") or "").strip().lower()
    ast_info = _safe_literal_eval(row.get("ast_info")) or {}
    dyn_info = _safe_literal_eval(row.get("dynamic_info")) or {}
    lib_info = _safe_literal_eval(row.get("lib_info")) or {}

    # AST parse errors
    try:
        if ast_info and (ast_info.get("ast_parsed") is False):
            for err in ast_info.get("ast_errors") or []:
                if isinstance(err, dict):
                    t = _clean_label(err.get("type", ""))
                    if t:
                        labels.add(t)
    except Exception:
        pass

    # Dynamic/runtime errors
    try:
        if isinstance(dyn_info, dict) and str(dyn_info.get("status", "") or "").strip().lower() == "failed":
            et = _clean_label(dyn_info.get("error_type", ""))
            if et:
                if et == "AssertionError":
                    labels.add("LogicalError")
                else:
                    labels.add(et)
    except Exception:
        pass

    # Lib API counters mapped to python-like names
    try:
        if isinstance(lib_info, dict) and lib_info.get("libapi_analyzed") is True:
            if int(lib_info.get("name_error", 0) or 0) > 0:
                labels.add("NameError")
            if int(lib_info.get("attribute_error", 0) or 0) > 0:
                labels.add("AttributeError")
            if int(lib_info.get("module_not_found", 0) or 0) > 0:
                labels.add("ModuleNotFoundError")

            # If libapi flagged errors but counters didn't map, keep a generic label
            if int(lib_info.get("total_libapi_errors", 0) or 0) > 0 and not (labels & {"NameError", "AttributeError", "ModuleNotFoundError"}):
                labels.add("LibAPIError")
    except Exception:
        pass

    # Guarantee at least one label for hallucinated rows
    if status == "hallucinated" and not labels:
        labels.add("UnknownHallucination")

    return labels


# count_{k,e}: (client -> error_label -> count)
count_ke = defaultdict(lambda: defaultdict(int))
for _, row in df.iterrows():
    k = row["_client"]
    if k not in adapters_found:
        continue
    for e in parse_error_labels(row):
        count_ke[k][e] += 1

# F_e = sum over k of count_{k,e}
F_e = defaultdict(int)
for k in adapters_found:
    for e, c in count_ke[k].items():
        F_e[e] += c

total_F = sum(F_e.values())
total_E = 0.0
if total_F == 0:
    alpha = {k: 1.0 / len(adapters_found) for k in adapters_found}
    print("No parsed errors in CSV; using uniform weights:", alpha)
else:
    weight_e = {e: F_e[e] / total_F for e in F_e}
    E_k = {k: sum(count_ke[k][e] * weight_e[e] for e in weight_e) for k in adapters_found}
    total_E = sum(E_k.values())
    if total_E == 0:
        alpha = {k: 1.0 / len(adapters_found) for k in adapters_found}
    else:
        alpha = {k: E_k[k] / total_E for k in adapters_found}
    print("Error-type weights (sample):", dict(list(weight_e.items())[:5]))
    print("Client error scores E_k:", E_k)
    print("Client weights alpha_k:", alpha)

Error-type weights (sample): {'LogicalError': 0.40492610837438425, 'SyntaxError': 0.03842364532019704, 'NameError': 0.21379310344827587, 'TypeError': 0.08275862068965517, 'ValueError': 0.06206896551724138}
Client error scores E_k: {'mbpp': 41.320197044334975, 'humaneval': 9.229556650246305, 'ds1000': 183.52807881773398}
Client weights alpha_k: {'mbpp': 0.1765233238912576, 'humaneval': 0.039429434864408704, 'ds1000': 0.7840472412443337}


In [3]:
# --- Tabulate counts (print before conversion/merge) ---
clients = list(adapters_found.keys())
all_errors = sorted({e for k in clients for e in count_ke[k].keys()})

counts_table = pd.DataFrame(
    [{"client": k, **{e: int(count_ke[k].get(e, 0)) for e in all_errors}} for k in clients]
).set_index("client")

error_totals = pd.Series({e: int(F_e.get(e, 0)) for e in all_errors}).sort_values(ascending=False)

print("\n=== Error labels discovered ===")
print(f"num_error_labels={len(all_errors)}")

print("\n=== Per-client error counts (non-zero only) ===")
for k in clients:
    nonzero = {e: c for e, c in count_ke[k].items() if c}
    nonzero_sorted = dict(sorted(nonzero.items(), key=lambda x: (-x[1], x[0])))
    print(f"\n[{k}] num_labels={len(nonzero_sorted)} total_counts={sum(nonzero_sorted.values())}")
    if nonzero_sorted:
        print(pd.Series(nonzero_sorted).head(50))

print("\n=== Per-client error count table (top 30 errors by total) ===")
top_errors = list(error_totals.head(30).index)
if top_errors:
    display(counts_table[top_errors])
else:
    display(counts_table)

print("\n=== Overall error totals (top 50) ===")
display(error_totals.head(50))

print("\nFinal alpha_k:", alpha)


=== Error labels discovered ===
num_error_labels=21

=== Per-client error counts (non-zero only) ===

[mbpp] num_labels=9 total_counts=165
LogicalError         74
NameError            42
SyntaxError          32
TypeError            10
ValueError            3
AttributeError        1
IndentationError      1
IndexError            1
UnboundLocalError     1
dtype: int64

[humaneval] num_labels=7 total_counts=31
LogicalError      20
NameError          4
SyntaxError        3
AttributeError     1
IndexError         1
TimeoutError       1
TypeError          1
dtype: int64

[ds1000] num_labels=20 total_counts=819
LogicalError              317
NameError                 171
TypeError                  73
IndentationError           68
ValueError                 60
AttributeError             57
KeyError                   19
TimeoutError               13
RuntimeError               11
NotFittedError              6
AxisError                   4
IndexError                  4
SyntaxError                 

,LogicalError,NameError,TypeError,IndentationError,ValueError,AttributeError,SyntaxError,KeyError,TimeoutError,RuntimeError,...,NotFittedError,AxisError,FileNotFoundError,IndexingError,NotImplementedError,ModuleNotFoundError,InvalidIndexError,LinAlgError,UnboundLocalError,UnidentifiedImageError
client,,,,,,,,,,,,,,,,,,,,,
mbpp,74,42,10,1,3,1,32,0,0,0,...,0,0,0,0,0,0,0,0,1,0
humaneval,20,4,1,0,0,1,3,0,1,0,...,0,0,0,0,0,0,0,0,0,0
ds1000,317,171,73,68,60,57,4,19,13,11,...,6,4,3,2,2,2,1,1,0,1



=== Overall error totals (top 50) ===


LogicalError              411
NameError                 217
TypeError                  84
IndentationError           69
ValueError                 63
AttributeError             59
SyntaxError                39
KeyError                   19
TimeoutError               14
RuntimeError               11
IndexError                  6
NotFittedError              6
AxisError                   4
FileNotFoundError           3
IndexingError               2
NotImplementedError         2
ModuleNotFoundError         2
InvalidIndexError           1
LinAlgError                 1
UnboundLocalError           1
UnidentifiedImageError      1
dtype: int64


Final alpha_k: {'mbpp': 0.1765233238912576, 'humaneval': 0.039429434864408704, 'ds1000': 0.7840472412443337}


## 3. Load adapter weights and compute weighted average

In [5]:
from safetensors.torch import load_file, save_file
import torch

state_dicts = {}
keys_ref = None
for name, path in adapters_found.items():
    p = path / "adapter_model.safetensors"
    state_dicts[name] = load_file(str(p))
    if keys_ref is None:
        keys_ref = set(state_dicts[name].keys())
    else:
        if set(state_dicts[name].keys()) != keys_ref:
            raise ValueError(f"Adapter keys differ: {name} vs reference")

print("Loaded", len(state_dicts), "adapters; number of keys:", len(keys_ref))

merged = {}
for key in keys_ref:
    merged[key] = sum(alpha[name] * state_dicts[name][key].float() for name in adapters_found)

print("Merged state dict computed.")

Loaded 3 adapters; number of keys: 504
Merged state dict computed.


## 4. Save merged adapter and config

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_file(merged, str(OUTPUT_DIR / "adapter_model.safetensors"))

first_client_path = adapters_found[list(adapters_found.keys())[0]]
config_src = first_client_path / "adapter_config.json"
if config_src.exists():
    shutil.copy2(str(config_src), str(OUTPUT_DIR / "adapter_config.json"))

for f in ["tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "README.md"]:
    src = first_client_path / f
    if src.exists():
        shutil.copy2(str(src), str(OUTPUT_DIR / f))

print("Saved to:", OUTPUT_DIR)

Saved to: d:\Desktop\MIT\CODES\FYP-26\FED-CONS-FINAL\aggregated_adapters\fedavg_error


## 5. Smoke load (optional)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(base_id, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
print("Smoke load OK. Merged adapter is ready for verification notebooks.")